# Seq2Seq (Encoder-Decoder) — English to French Translation
### A Complete PyTorch Implementation in Google Colab

---

## Introduction

The **Encoder-Decoder** (Seq2Seq) architecture is a deep learning framework designed to map one  
sequence to another sequence of potentially different length. It was introduced by Sutskever et al. (2014)  
and revolutionized machine translation, text summarization, chatbots, and speech recognition.

**Key insight:** Split the learning problem into two sub-networks:
- **Encoder** — reads and compresses the input into a fixed-size *context vector*
- **Decoder** — unpacks the context vector into the target output, step by step

**This notebook:**
- Task: English → French translation
- Training data: 50 hand-crafted sentence pairs (defined below)
- Model: 2-layer LSTM Encoder-Decoder with Teacher Forcing
- Framework: PyTorch

---


## Required Libraries

| Library | Purpose | Pre-installed in Colab? |
|---------|---------|------------------------|
| `torch` | Build and train neural networks | Yes |
| `numpy` | Numerical operations | Yes |
| `matplotlib` | Plot training loss | Yes |
| `random`, `time` | Shuffling, timing | Built-in |

> No `pip install` needed — all libraries are pre-installed in Google Colab.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import time
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == 'cuda':
    print(f"GPU             : {torch.cuda.get_device_name(0)}")


---
## Training Data — 50 English → French Sentence Pairs

The dataset covers 5 categories:
1. **Greetings & Farewells** (10 pairs)
2. **Basic Phrases** (10 pairs)
3. **Questions** (10 pairs)
4. **Personal Expressions** (10 pairs)
5. **Simple Sentences** (10 pairs)

All text is lowercase and punctuation-free for simplicity.


In [ ]:
# ── Training Data ────────────────────────────────────────────────────────
# 50 English → French pairs (word-level, lowercase, no punctuation)

pairs = [
    # ── Category 1: Greetings & Farewells ─────────────────────────────
    ("hello",                    "bonjour"),
    ("hi",                       "salut"),
    ("goodbye",                  "au revoir"),
    ("good morning",             "bonjour"),
    ("good evening",             "bonsoir"),
    ("good night",               "bonne nuit"),
    ("welcome",                  "bienvenue"),
    ("see you later",            "a bientot"),
    ("how are you",              "comment allez vous"),
    ("i am fine",                "je vais bien"),

    # ── Category 2: Basic Phrases ─────────────────────────────────────
    ("thank you",                "merci"),
    ("you are welcome",          "de rien"),
    ("please",                   "s il vous plait"),
    ("excuse me",                "excusez moi"),
    ("sorry",                    "pardon"),
    ("yes",                      "oui"),
    ("no",                       "non"),
    ("i do not know",            "je ne sais pas"),
    ("i understand",             "je comprends"),
    ("i do not understand",      "je ne comprends pas"),

    # ── Category 3: Questions ─────────────────────────────────────────
    ("what is your name",        "quel est votre nom"),
    ("where are you from",       "d ou venez vous"),
    ("how old are you",          "quel age avez vous"),
    ("what time is it",          "quelle heure est il"),
    ("where is the hotel",       "ou est l hotel"),
    ("do you speak english",     "parlez vous anglais"),
    ("how much does it cost",    "combien ca coute"),
    ("where is the bathroom",    "ou sont les toilettes"),
    ("can you help me",          "pouvez vous m aider"),
    ("what is this",             "qu est ce que c est"),

    # ── Category 4: Personal Expressions ──────────────────────────────
    ("my name is john",          "je m appelle john"),
    ("i am hungry",              "j ai faim"),
    ("i am thirsty",             "j ai soif"),
    ("i am tired",               "je suis fatigue"),
    ("i am happy",               "je suis heureux"),
    ("i am lost",                "je suis perdu"),
    ("i love france",            "j aime la france"),
    ("i like coffee",            "j aime le cafe"),
    ("i speak a little french",  "je parle un peu francais"),
    ("the food is good",         "la nourriture est bonne"),

    # ── Category 5: Simple Sentences ──────────────────────────────────
    ("the cat is black",         "le chat est noir"),
    ("the dog is small",         "le chien est petit"),
    ("the book is on the table", "le livre est sur la table"),
    ("it is a beautiful day",    "c est une belle journee"),
    ("i want to go to paris",    "je veux aller a paris"),
    ("red",                      "rouge"),
    ("blue",                     "bleu"),
    ("green",                    "vert"),
    ("one two three",            "un deux trois"),
    ("four five six",            "quatre cinq six"),
]

print(f"Total training pairs : {len(pairs)}")
print()
print(f"{'English':<30} {'French':<30}")
print("-" * 60)
for en, fr in pairs[:10]:
    print(f"{en:<30} {fr:<30}")
print(f"  ... ({len(pairs) - 10} more pairs)")


---
## Vocabulary Building

We build two separate vocabularies — one for English (source) and one for French (target).  
Each vocabulary maps words ↔ integer indices, with 3 special tokens reserved:

| Token | Index | Meaning |
|-------|-------|---------|
| `<PAD>` | 0 | Padding (unused in this implementation) |
| `<SOS>` | 1 | Start-of-Sequence (fed to decoder as first input) |
| `<EOS>` | 2 | End-of-Sequence (signals decoder to stop) |


In [ ]:
# ── Special Token Indices ─────────────────────────────────────────────────
PAD_token = 0
SOS_token = 1
EOS_token = 2

class Vocabulary:
    """
    Word-level vocabulary: maps words <-> integer indices.
    Automatically handles unknown words via 'add_word'.
    """
    def __init__(self, name):
        self.name = name
        self.word2index = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
        self.index2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>'}
        self.word_count  = {}
        self.n_words     = 3   # starts after special tokens

    def add_sentence(self, sentence):
        for word in sentence.split():
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word]        = self.n_words
            self.index2word[self.n_words] = word
            self.word_count[word]        = 1
            self.n_words                += 1
        else:
            self.word_count[word] += 1


# Build source and target vocabularies from all pairs
en_vocab = Vocabulary("English")
fr_vocab = Vocabulary("French")

for en, fr in pairs:
    en_vocab.add_sentence(en)
    fr_vocab.add_sentence(fr)

print(f"English vocabulary size : {en_vocab.n_words:3d} words")
print(f"French  vocabulary size : {fr_vocab.n_words:3d} words")
print()
print("Sample English tokens (index → word):")
for idx in range(3, 11):
    print(f"  [{idx}] {en_vocab.index2word[idx]}")
print()
print("Sample French tokens (index → word):")
for idx in range(3, 11):
    print(f"  [{idx}] {fr_vocab.index2word[idx]}")


---
## Sentence → Tensor Conversion

Before feeding text into the model, each sentence is:
1. Split into words
2. Each word replaced by its index in the vocabulary
3. `<EOS>` appended to signal end-of-sequence
4. Wrapped in a PyTorch tensor shaped `[seq_len, 1]` (batch_size=1)


In [ ]:
# ── Tensor Helpers ───────────────────────────────────────────────────────

def sentence_to_tensor(vocab, sentence):
    """
    Convert a sentence string into a LongTensor of word indices.
    Appends EOS token at the end.
    Shape: [seq_len + 1, 1]
    """
    indices = [vocab.word2index[w] for w in sentence.split()]
    indices.append(EOS_token)
    return torch.tensor(indices, dtype=torch.long, device=device).view(-1, 1)

def pair_to_tensors(pair):
    """Return (input_tensor, target_tensor) for an (en, fr) pair."""
    return sentence_to_tensor(en_vocab, pair[0]), \
           sentence_to_tensor(fr_vocab, pair[1])

# ── Demonstrate ───────────────────────────────────────────────────────────
sample_pair = ("how are you", "comment allez vous")
src_t, trg_t = pair_to_tensors(sample_pair)

print(f"Input sentence  : '{sample_pair[0]}'")
print(f"Input indices   : {src_t.squeeze().tolist()}  (last = EOS={EOS_token})")
print()
print(f"Target sentence : '{sample_pair[1]}'")
print(f"Target indices  : {trg_t.squeeze().tolist()}  (last = EOS={EOS_token})")


---
## Part 1 — Encoder

The **Encoder** is a multi-layer LSTM that reads the entire input sequence  
and compresses it into a pair of tensors `(hidden, cell)` — the **context vector**.

```
Input tokens:  ["how", "are", "you", <EOS>]
                  ↓        ↓       ↓     ↓
              [Embed] [Embed] [Embed] [Embed]
                  ↓        ↓       ↓     ↓
              [LSTM₁]─[LSTM₂]─[LSTM₃]─[LSTM₄]  ← Layer 1
                  ↓        ↓       ↓     ↓
              [LSTM₁]─[LSTM₂]─[LSTM₃]─[LSTM₄]  ← Layer 2
                                              ↓
                                       (hT, cT)   ← Context Vector
```

The LSTM passes hidden state `h_t` and cell state `c_t` from step to step.  
Only the **final** hidden and cell states are passed to the decoder.


In [ ]:
# ── Encoder ──────────────────────────────────────────────────────────────

class Encoder(nn.Module):
    """
    Multi-layer LSTM Encoder.

    Flow:
        input_tokens → Embedding → LSTM (T steps) → (hidden_T, cell_T)

    Parameters
    ----------
    input_size   : source vocabulary size
    embedding_dim: size of word embedding vectors
    hidden_dim   : LSTM hidden state size
    num_layers   : number of stacked LSTM layers
    dropout      : dropout probability between layers
    """

    def __init__(self, input_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Maps word index → dense embedding vector
        self.embedding = nn.Embedding(input_size, embedding_dim, padding_idx=PAD_token)

        # Stacked LSTM: processes one token per timestep
        self.lstm = nn.LSTM(
            input_size  = embedding_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0.0,
            batch_first = False   # [seq_len, batch, features]
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        """
        src : [seq_len, batch_size]  — source token indices
        Returns
        -------
        hidden : [num_layers, batch, hidden_dim]  — final hidden state
        cell   : [num_layers, batch, hidden_dim]  — final cell state
        """
        # 1. Embed:  [seq_len, batch] → [seq_len, batch, embed_dim]
        embedded = self.dropout(self.embedding(src))

        # 2. LSTM:   returns ALL hidden states + final (hidden, cell)
        #    outputs : [seq_len, batch, hidden_dim]  — not used here
        #    hidden  : [num_layers, batch, hidden_dim]
        #    cell    : [num_layers, batch, hidden_dim]
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Return only the context vector (final state)
        return hidden, cell


# ── Quick sanity check ────────────────────────────────────────────────────
test_enc = Encoder(en_vocab.n_words, 128, 256, 2, 0.3).to(device)
test_src, _ = pair_to_tensors(pairs[0])
h, c = test_enc(test_src)

print("Encoder forward-pass check:")
print(f"  Input shape          : {test_src.shape}  (seq_len=1, batch=1)")
print(f"  Output hidden shape  : {h.shape}          (layers, batch, hidden)")
print(f"  Output cell   shape  : {c.shape}")
print()
print(test_enc)
total_enc = sum(p.numel() for p in test_enc.parameters() if p.requires_grad)
print(f"\nEncoder trainable params: {total_enc:,}")


---
## Part 1 — Decoder

The **Decoder** is also an LSTM, but it works differently:
- It is **initialized** with the encoder's final `(hidden, cell)` — the context vector
- It generates **one token per step**, using its own previous output as the next input
- It stops when it generates `<EOS>` (during inference) or when the target is exhausted (during training)

```
Context (hT, cT)  ←─── from Encoder
       ↓
    <SOS>  →  [Dec LSTM] → "comment" → [Dec LSTM] → "allez" → [Dec LSTM] → "vous" → <EOS>
                  ↑                         ↑                      ↑
            [FC → Softmax]           [FC → Softmax]         [FC → Softmax]
```

### Teacher Forcing
During **training**, instead of feeding the model's (possibly wrong) prediction as the next input,  
we feed the **actual ground truth token**. This:
- Prevents error accumulation during training
- Speeds up convergence significantly
- Is disabled during inference (free-running decoding)


In [ ]:
# ── Decoder ──────────────────────────────────────────────────────────────

class Decoder(nn.Module):
    """
    Multi-layer LSTM Decoder with FC output projection.

    Flow (one step):
        prev_token → Embedding → LSTM (1 step) → FC layer → logits

    The (hidden, cell) from the previous step carry forward temporal context.
    At the first step, (hidden, cell) come from the Encoder.

    Parameters
    ----------
    output_size  : target vocabulary size
    embedding_dim: size of word embeddings
    hidden_dim   : LSTM hidden state size (must match Encoder)
    num_layers   : number of stacked LSTM layers (must match Encoder)
    dropout      : dropout probability
    """

    def __init__(self, output_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.output_size = output_size
        self.hidden_dim  = hidden_dim
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(output_size, embedding_dim, padding_idx=PAD_token)

        self.lstm = nn.LSTM(
            input_size  = embedding_dim,
            hidden_size = hidden_dim,
            num_layers  = num_layers,
            dropout     = dropout if num_layers > 1 else 0.0,
            batch_first = False
        )

        # Projects hidden state → vocabulary logits
        self.fc_out = nn.Linear(hidden_dim, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token, hidden, cell):
        """
        token  : [1, batch_size]                — current input token index
        hidden : [num_layers, batch, hidden_dim] — previous hidden state
        cell   : [num_layers, batch, hidden_dim] — previous cell state

        Returns
        -------
        logits : [batch, output_size]            — unnormalized scores for next token
        hidden : [num_layers, batch, hidden_dim] — updated hidden state
        cell   : [num_layers, batch, hidden_dim] — updated cell state
        """
        # token [1, batch] → [1, batch, embed_dim]
        embedded = self.dropout(self.embedding(token))

        # One LSTM step
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        # output: [1, batch, hidden_dim]

        # Project to vocab: [batch, hidden_dim] → [batch, output_size]
        logits = self.fc_out(output.squeeze(0))

        return logits, hidden, cell


# ── Quick check ───────────────────────────────────────────────────────────
test_dec  = Decoder(fr_vocab.n_words, 128, 256, 2, 0.3).to(device)
sos_input = torch.tensor([[SOS_token]], device=device)   # [1, 1]
logits, h2, c2 = test_dec(sos_input, h, c)

print("Decoder forward-pass check:")
print(f"  Input token shape : {sos_input.shape}  (1 token, batch=1)")
print(f"  Output logits     : {logits.shape}     (batch=1, vocab=fr_vocab.n_words)")
print(f"  Output hidden     : {h2.shape}")
print()
print(test_dec)
total_dec = sum(p.numel() for p in test_dec.parameters() if p.requires_grad)
print(f"\nDecoder trainable params: {total_dec:,}")


---
## Part 2 — Seq2Seq Wrapper

The `Seq2Seq` class wires Encoder + Decoder together and manages:
- Passing the context vector from encoder to decoder
- The teacher forcing loop over all target tokens
- Storing all decoder predictions for loss computation


In [ ]:
# ── Seq2Seq Model ─────────────────────────────────────────────────────────

class Seq2Seq(nn.Module):
    """
    Complete Sequence-to-Sequence model.

    Encoding:
        src → Encoder → (hidden_T, cell_T)

    Decoding loop (t = 1 .. trg_len):
        input_t → Decoder(hidden_t-1, cell_t-1) → logits_t
        if teacher_forcing:
            input_(t+1) = trg[t]          ← ground truth
        else:
            input_(t+1) = argmax(logits_t) ← model's own prediction
    """

    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

        # Sanity: encoder and decoder hidden dims must match
        assert encoder.hidden_dim == decoder.hidden_dim, \
            "Encoder/Decoder hidden_dim mismatch!"
        assert encoder.num_layers == decoder.num_layers, \
            "Encoder/Decoder num_layers mismatch!"

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        """
        src : [src_len, batch]   — source indices
        trg : [trg_len, batch]   — target indices (incl. SOS at trg[0])
        teacher_forcing_ratio : probability of using ground truth as next input

        Returns
        -------
        outputs : [trg_len, batch, trg_vocab_size]
        """
        trg_len        = trg.shape[0]
        batch_size     = trg.shape[1]
        trg_vocab_size = self.decoder.output_size

        # Buffer for all decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size, device=self.device)

        # ── Encode ───────────────────────────────────────────────────────
        hidden, cell = self.encoder(src)     # context vector

        # ── Decode ───────────────────────────────────────────────────────
        dec_input = trg[0].unsqueeze(0)       # first input = <SOS>

        for t in range(1, trg_len):
            logits, hidden, cell = self.decoder(dec_input, hidden, cell)
            outputs[t] = logits

            # Teacher forcing coin flip
            if random.random() < teacher_forcing_ratio:
                dec_input = trg[t].unsqueeze(0)     # ground truth
            else:
                dec_input = logits.argmax(1).unsqueeze(0)  # model prediction

        return outputs


---
## Hyperparameters & Model Initialization


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────

EMBEDDING_DIM         = 128    # word embedding size
HIDDEN_DIM            = 256    # LSTM hidden state size
NUM_LAYERS            = 2      # stacked LSTM layers
DROPOUT               = 0.3    # dropout probability
LEARNING_RATE         = 0.001  # Adam learning rate
N_EPOCHS              = 300    # training epochs
TEACHER_FORCING_RATIO = 0.5    # 50% teacher forcing during training
CLIP                  = 1.0    # gradient clipping norm

# ── Build Models ──────────────────────────────────────────────────────────

encoder = Encoder(
    input_size    = en_vocab.n_words,
    embedding_dim = EMBEDDING_DIM,
    hidden_dim    = HIDDEN_DIM,
    num_layers    = NUM_LAYERS,
    dropout       = DROPOUT
).to(device)

decoder = Decoder(
    output_size   = fr_vocab.n_words,
    embedding_dim = EMBEDDING_DIM,
    hidden_dim    = HIDDEN_DIM,
    num_layers    = NUM_LAYERS,
    dropout       = DROPOUT
).to(device)

model = Seq2Seq(encoder, decoder, device).to(device)

# ── Weight Initialization ─────────────────────────────────────────────────
# Uniform [-0.08, 0.08] initialization helps training stability
def init_weights(m):
    for name, param in m.named_parameters():
        nn.init.uniform_(param.data, -0.08, 0.08)

model.apply(init_weights)

# ── Optimizer & Loss ──────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_token)

# ── Summary ───────────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
enc_params   = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
dec_params   = sum(p.numel() for p in decoder.parameters() if p.requires_grad)

print("=" * 50)
print("  Model Summary")
print("=" * 50)
print(f"  Encoder params  : {enc_params:>8,}")
print(f"  Decoder params  : {dec_params:>8,}")
print(f"  Total params    : {total_params:>8,}")
print(f"  Embedding dim   : {EMBEDDING_DIM}")
print(f"  Hidden dim      : {HIDDEN_DIM}")
print(f"  LSTM layers     : {NUM_LAYERS}")
print(f"  Dropout         : {DROPOUT}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Teacher forcing : {TEACHER_FORCING_RATIO}")
print("=" * 50)


---
## Training Loop

For each epoch:
1. **Shuffle** the training pairs (prevents order-based overfitting)
2. For each pair, run a **forward pass** through Seq2Seq
3. Compute **CrossEntropy loss** between predicted and target tokens
4. **Backpropagate** gradients
5. **Clip gradients** (prevents exploding gradients in RNNs)
6. **Update** weights with Adam optimizer


In [ ]:
# ── Training Function ─────────────────────────────────────────────────────

def train_one_epoch(model, pairs, optimizer, criterion, teacher_forcing_ratio, clip):
    """Train model over all pairs for one epoch. Returns mean loss."""
    model.train()
    epoch_loss = 0.0
    random.shuffle(pairs)

    for pair in pairs:
        src_tensor, trg_tensor = pair_to_tensors(pair)

        optimizer.zero_grad()

        # Forward
        output = model(src_tensor, trg_tensor, teacher_forcing_ratio)
        # output: [trg_len, 1, vocab_size]

        # Flatten for cross-entropy (skip index 0 = SOS slot)
        output_dim  = output.shape[-1]
        output_flat = output[1:].view(-1, output_dim)   # [(trg_len-1), vocab]
        trg_flat    = trg_tensor[1:].view(-1)           # [(trg_len-1)]

        loss = criterion(output_flat, trg_flat)
        loss.backward()

        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(pairs)


# ── Run Training ──────────────────────────────────────────────────────────
print("=" * 55)
print("  Training Seq2Seq: English → French")
print("=" * 55)
print(f"  Pairs     : {len(pairs)}")
print(f"  Epochs    : {N_EPOCHS}")
print(f"  Device    : {device}")
print("=" * 55)

loss_history = []
start_time   = time.time()

for epoch in range(1, N_EPOCHS + 1):
    loss = train_one_epoch(
        model, pairs, optimizer, criterion,
        TEACHER_FORCING_RATIO, CLIP
    )
    loss_history.append(loss)

    if epoch == 1 or epoch % 50 == 0:
        elapsed = time.time() - start_time
        print(f"  Epoch [{epoch:3d}/{N_EPOCHS}]  Loss: {loss:.4f}  "
              f"Elapsed: {elapsed:.1f}s")

print()
print(f"Training complete! Final loss: {loss_history[-1]:.4f}")


---
## Inference — Translation Function

During inference:
- **No teacher forcing** — the model uses its own predictions as next input
- **Greedy decoding** — pick the token with the highest probability at each step
- Stop when `<EOS>` is predicted or `max_len` is reached


In [ ]:
# ── Translate Function ────────────────────────────────────────────────────

def translate(model, sentence, max_len=20, verbose=False):
    """
    Translate an English sentence to French using greedy decoding.

    Parameters
    ----------
    model    : trained Seq2Seq model
    sentence : English input string (lowercase, no punctuation)
    max_len  : maximum output length
    verbose  : if True, print step-by-step decoding

    Returns
    -------
    str : translated French sentence
    """
    model.eval()

    with torch.no_grad():
        # Tokenize & encode source
        try:
            src_tensor = sentence_to_tensor(en_vocab, sentence.lower())
        except KeyError as e:
            return f"[Unknown word: {e}]"

        # Encode: get context vector
        hidden, cell = model.encoder(src_tensor)

        if verbose:
            print(f"Input       : '{sentence}'")
            print(f"Encoded to  : hidden {hidden.shape}, cell {cell.shape}")
            print()

        # Decode: start from <SOS>
        dec_input     = torch.tensor([[SOS_token]], device=device)
        translated    = []

        for step in range(max_len):
            logits, hidden, cell = model.decoder(dec_input, hidden, cell)
            probs     = torch.softmax(logits, dim=-1)
            pred_idx  = logits.argmax(1).item()
            pred_word = fr_vocab.index2word.get(pred_idx, '<UNK>')
            pred_prob = probs[0, pred_idx].item()

            if verbose:
                print(f"  Step {step+1}: '{pred_word}'  (prob={pred_prob:.3f})")

            if pred_idx == EOS_token:
                break

            translated.append(pred_word)
            dec_input = torch.tensor([[pred_idx]], device=device)

    return ' '.join(translated)


# ── Demonstrate verbose decoding ──────────────────────────────────────────
print("Verbose decoding example:")
print("-" * 40)
_ = translate(model, "how are you", verbose=True)
print()
print("Result:", translate(model, "how are you"))


---
## Translation Results — Model Output


In [ ]:
# ── Full Translation Test ─────────────────────────────────────────────────

test_cases = [
    ("hello",                    "bonjour"),
    ("thank you",                "merci"),
    ("how are you",              "comment allez vous"),
    ("i am fine",                "je vais bien"),
    ("good morning",             "bonjour"),
    ("good night",               "bonne nuit"),
    ("i love france",            "j aime la france"),
    ("i am hungry",              "j ai faim"),
    ("where is the hotel",       "ou est l hotel"),
    ("what is your name",        "quel est votre nom"),
    ("the cat is black",         "le chat est noir"),
    ("i want to go to paris",    "je veux aller a paris"),
    ("do you speak english",     "parlez vous anglais"),
    ("it is a beautiful day",    "c est une belle journee"),
    ("the book is on the table", "le livre est sur la table"),
]

print("=" * 80)
print(f"  {'English Input':<28} {'Expected (FR)':<26} {'Predicted (FR)':<22} OK?")
print("=" * 80)

correct = 0
for eng, expected in test_cases:
    predicted = translate(model, eng)
    ok        = predicted.strip() == expected.strip()
    correct  += int(ok)
    mark      = "✓" if ok else "✗"
    print(f"  {eng:<28} {expected:<26} {predicted:<22} {mark}")

print("=" * 80)
accuracy = 100 * correct / len(test_cases)
print(f"  Accuracy: {correct}/{len(test_cases)} ({accuracy:.0f}%)")
print()

# ── Test unseen combinations ──────────────────────────────────────────────
print("Testing generalization (unseen inputs):")
print("-" * 50)
generalize_tests = [
    "i am happy",
    "blue",
    "one two three",
    "sorry",
    "welcome",
]
for eng in generalize_tests:
    pred = translate(model, eng)
    print(f"  '{eng}' → '{pred}'")


---
## Training Loss Curve


In [ ]:
# ── Plot Training Loss ────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full loss curve
axes[0].plot(loss_history, color='steelblue', linewidth=1.2, alpha=0.8)
axes[0].set_title('Training Loss (all epochs)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].grid(True, alpha=0.3)

# Smoothed loss (moving average)
window = 10
smoothed = np.convolve(loss_history, np.ones(window)/window, mode='valid')
axes[1].plot(smoothed, color='darkorange', linewidth=1.5)
axes[1].set_title(f'Smoothed Loss (window={window})', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Cross-Entropy Loss')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Seq2Seq English→French | Loss over Training', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('seq2seq_loss.png', dpi=120, bbox_inches='tight')
plt.show()
print("Loss plot saved as 'seq2seq_loss.png'")
print(f"Loss: start={loss_history[0]:.4f}  →  end={loss_history[-1]:.4f}")
print(f"Improvement: {loss_history[0] - loss_history[-1]:.4f} ({100*(loss_history[0]-loss_history[-1])/loss_history[0]:.1f}% reduction)")


---
## Save & Load Model


In [ ]:
# ── Save ─────────────────────────────────────────────────────────────────

torch.save({
    'model_state_dict'  : model.state_dict(),
    'en_vocab_w2i'      : en_vocab.word2index,
    'en_vocab_i2w'      : en_vocab.index2word,
    'fr_vocab_w2i'      : fr_vocab.word2index,
    'fr_vocab_i2w'      : fr_vocab.index2word,
    'hyperparams': {
        'EMBEDDING_DIM' : EMBEDDING_DIM,
        'HIDDEN_DIM'    : HIDDEN_DIM,
        'NUM_LAYERS'    : NUM_LAYERS,
        'DROPOUT'       : DROPOUT,
    }
}, 'seq2seq_en_fr.pt')

print("Model saved  →  seq2seq_en_fr.pt")
print()
print("To reload later:")
print("""
  checkpoint = torch.load('seq2seq_en_fr.pt')
  # Rebuild vocabs and model, then:
  model.load_state_dict(checkpoint['model_state_dict'])
""")

# ── Final architecture summary ─────────────────────────────────────────────
print("=" * 50)
print("  Final Architecture Summary")
print("=" * 50)
print(f"  Source vocab (EN)  : {en_vocab.n_words} words")
print(f"  Target vocab (FR)  : {fr_vocab.n_words} words")
print(f"  Embedding dim      : {EMBEDDING_DIM}")
print(f"  LSTM hidden dim    : {HIDDEN_DIM}")
print(f"  LSTM layers        : {NUM_LAYERS}")
print(f"  Teacher forcing    : {TEACHER_FORCING_RATIO}")
print(f"  Total parameters   : {total_params:,}")
print(f"  Final train loss   : {loss_history[-1]:.4f}")
print("=" * 50)
